## data leakage 수정본
- train_test_split을 영화 단위로

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np

# 파일 경로 확인
X = np.load('/content/drive/MyDrive/ColabNotebooks/dataEng/DE_PJ2/X_clips_local.npy')
y_str = np.load('/content/drive/MyDrive/ColabNotebooks/dataEng/DE_PJ2/y_clips_local.npy')
movie_ids = np.load('/content/drive/MyDrive/ColabNotebooks/dataEng/DE_PJ2/movie_ids_local.npy')

print(f'X: {X.shape}')
print(f'violence: {sum(y_str=="violence")}개')
print(f'neg_easy: {sum(y_str=="neg_easy")}개')

Mounted at /content/drive
X: (2606, 4, 2048)
violence: 1056개
neg_easy: 1550개


In [2]:
!pip install torch torchvision scikit-learn -q

In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader

In [4]:
LABEL_MAP = {'neg_easy': 0, 'violence': 1}
y = np.array([LABEL_MAP[l] for l in y_str], dtype=np.int64)

# 영화별 정규화
X_norm = X.copy()
for movie in set(movie_ids):
    idx = np.where(movie_ids == movie)[0]
    if len(idx) > 1:
        orig_shape = X[idx].shape
        flat = X[idx].reshape(len(idx), -1)
        flat = StandardScaler().fit_transform(flat)
        X_norm[idx] = flat.reshape(orig_shape)

print('정규화 완료')


# train/val split - 영화 단위로
unique_movies = list(set(movie_ids))
np.random.seed(42)
np.random.shuffle(unique_movies)
n = len(unique_movies)
train_movies = unique_movies[:int(n*0.8)]
val_movies = unique_movies[int(n*0.8):]

train_idx = np.where(np.isin(movie_ids, train_movies))[0]
val_idx = np.where(np.isin(movie_ids, val_movies))[0]

X_train, y_train = X_norm[train_idx], y[train_idx]
X_val, y_val = X_norm[val_idx], y[val_idx]

print(f'train: {len(X_train)}개 ({len(train_movies)}개 영화)')
print(f'val: {len(X_val)}개 ({len(val_movies)}개 영화)')

class ClipDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(ClipDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(ClipDataset(X_val, y_val), batch_size=32)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')

class ViolenceLSTM(nn.Module):
    def __init__(self, input_size=2048, hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 2)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = ViolenceLSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())

    acc = sum(p==l for p,l in zip(all_preds, all_labels)) / len(all_labels)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {acc:.4f}')

print('\n--- classification report ---')
print(classification_report(all_labels, all_preds, target_names=['neg_easy', 'violence']))

torch.save(model.state_dict(), '/content/drive/MyDrive/ColabNotebooks/dataEng/DE_PJ2/violence_lstm_norm.pth')
print('모델 저장 완료')

정규화 완료
train: 1495개 (8개 영화)
val: 376개 (2개 영화)
디바이스: cuda
Epoch 1/30 | Loss: 0.3570 | Val Acc: 0.6170
Epoch 2/30 | Loss: 0.0813 | Val Acc: 0.6011
Epoch 3/30 | Loss: 0.0401 | Val Acc: 0.6410
Epoch 4/30 | Loss: 0.0166 | Val Acc: 0.6516
Epoch 5/30 | Loss: 0.0226 | Val Acc: 0.6702
Epoch 6/30 | Loss: 0.0252 | Val Acc: 0.6356
Epoch 7/30 | Loss: 0.0190 | Val Acc: 0.6436
Epoch 8/30 | Loss: 0.0101 | Val Acc: 0.6277
Epoch 9/30 | Loss: 0.0147 | Val Acc: 0.6277
Epoch 10/30 | Loss: 0.0137 | Val Acc: 0.6223
Epoch 11/30 | Loss: 0.0074 | Val Acc: 0.6170
Epoch 12/30 | Loss: 0.0423 | Val Acc: 0.6356
Epoch 13/30 | Loss: 0.0283 | Val Acc: 0.7021
Epoch 14/30 | Loss: 0.0198 | Val Acc: 0.6064
Epoch 15/30 | Loss: 0.0079 | Val Acc: 0.5957
Epoch 16/30 | Loss: 0.0120 | Val Acc: 0.6064
Epoch 17/30 | Loss: 0.0135 | Val Acc: 0.6037
Epoch 18/30 | Loss: 0.0016 | Val Acc: 0.5984
Epoch 19/30 | Loss: 0.0104 | Val Acc: 0.5638
Epoch 20/30 | Loss: 0.0071 | Val Acc: 0.6197
Epoch 21/30 | Loss: 0.0013 | Val Acc: 0.6037
Epoch 2

## 테스트

In [5]:
# 1. 설치
!pip install torch torchvision huggingface_hub scikit-learn -q

# 2. import
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import re
from huggingface_hub import hf_hub_download
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from google.colab import drive

# 3. Drive 마운트
drive.mount('/content/drive')

# 4. 디바이스
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')

# 5. ResNet
resnet = models.resnet50(weights='IMAGENET1K_V1')
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def extract_cnn_feature(image_path):
    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(x).squeeze().cpu().numpy()
    return feat

# 6. LSTM 모델
class ViolenceLSTM(nn.Module):
    def __init__(self, input_size=2048, hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 2)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = ViolenceLSTM().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/ColabNotebooks/dataEng/DE_PJ2/violence_lstm_norm.pth', map_location=device))
model.eval()
print('모델 로드 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
디바이스: cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 207MB/s]


모델 로드 완료


In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
def build_test_clips_sampled(movie_id, clip_len=4, max_clips=50):
    txt_path = hf_hub_download(
        repo_id='DEteam4/datasetVer3',
        filename=f'annotations/raw_txt/{movie_id}.txt',
        repo_type='dataset'
    )
    scenes = []
    with open(txt_path, 'r') as f:
        for line in f:
            match = re.match(r'\[(.+),\s*(\d+),\s*(\d+),\s*(\d+),\s*(\w+)\]', line.strip())
            if match:
                start = int(match.group(3))
                end = int(match.group(4))
                label = match.group(5)
                if label in ['violence', 'neg_easy']:
                    scenes.append((start, end, label))

    vio_scenes = [s for s in scenes if s[2] == 'violence']
    neg_scenes = [s for s in scenes if s[2] == 'neg_easy']
    sampled = vio_scenes[:max_clips//2] + neg_scenes[:max_clips//2]

    clips, clip_labels = [], []
    for (start, end, label) in sampled:
        mid = (start + end) // 2
        clip_frames = list(range(mid, mid + clip_len))
        clip_feats = []
        for f in clip_frames:
            try:
                img_path = hf_hub_download(
                    repo_id='DEteam4/datasetVer3',
                    filename=f'frames/{movie_id}/frame_{f:06d}.jpg',
                    repo_type='dataset'
                )
                clip_feats.append(extract_cnn_feature(img_path))
            except:
                clip_feats.append(np.zeros(2048))
        if len(clip_feats) == clip_len:
            clips.append(clip_feats)
            clip_labels.append(label)

    return clips, clip_labels

# 테스트 데이터 수집
test_clips, test_labels = [], []
for movie in ['Asura', '1YAwZ5CWXCQ']:
    print(f'{movie} 처리 중...')
    clips, labels = build_test_clips_sampled(movie, max_clips=50)
    test_clips.extend(clips)
    test_labels.extend(labels)
    print(f'{movie}: {len(clips)}개')

# 영화별 정규화
X_test = np.array(test_clips, dtype=np.float32)
movie_test = ['Asura'] * len(test_clips[:len(test_clips)//2]) + ['1YAwZ5CWXCQ'] * len(test_clips[len(test_clips)//2:])

# Asura, 1YAwZ5CWXCQ 각각 정규화
for movie in ['Asura', '1YAwZ5CWXCQ']:
    idx = [i for i, m in enumerate(movie_test) if m == movie]
    if len(idx) > 1:
        orig_shape = X_test[idx].shape
        flat = X_test[idx].reshape(len(idx), -1)
        flat = StandardScaler().fit_transform(flat)
        X_test[idx] = flat.reshape(orig_shape)

# 예측
LABEL_MAP = {'neg_easy': 0, 'violence': 1}
y_test = np.array([LABEL_MAP[l] for l in test_labels])

X_tensor = torch.tensor(X_test).to(device)
with torch.no_grad():
    preds = model(X_tensor).argmax(1).cpu().numpy()

print(classification_report(y_test, preds, target_names=['neg_easy', 'violence']))

Asura 처리 중...


Asura.txt:   0%|          | 0.00/2.01k [00:00<?, ?B/s]

frames/Asura/frame_000091.jpg:   0%|          | 0.00/55.0k [00:00<?, ?B/s]

frames/Asura/frame_000092.jpg:   0%|          | 0.00/53.3k [00:00<?, ?B/s]

frames/Asura/frame_000093.jpg:   0%|          | 0.00/52.4k [00:00<?, ?B/s]

...

frames/1YAwZ5CWXCQ/frame_001485.jpg:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

frames/1YAwZ5CWXCQ/frame_001486.jpg:   0%|          | 0.00/40.6k [00:00<?, ?B/s]

frames/1YAwZ5CWXCQ/frame_001487.jpg:   0%|          | 0.00/39.8k [00:00<?, ?B/s]

frames/1YAwZ5CWXCQ/frame_001488.jpg:   0%|          | 0.00/39.5k [00:00<?, ?B/s]

1YAwZ5CWXCQ: 47개
              precision    recall  f1-score   support

    neg_easy       0.59      0.86      0.70        50
    violence       0.71      0.36      0.48        47

    accuracy                           0.62        97
   macro avg       0.65      0.61      0.59        97
weighted avg       0.65      0.62      0.59        97

